# Chapter 5: Common Three-Dimensional Coordinate Systems

Companion notebook to Section 5 of the study guide (Kundu, Cohen & Dowling 2016, Appendix B.5).
Write your own notes under each heading as you study.

Before committing: **Restart** the kernel, then **Run All**, then **save**, so the notebook runs from top to bottom.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import trapezoid

from ch05_tools import (area_weighted_mean, coriolis_parameter, derivative_in_basis,
                        great_circle_distance, latlon_cell_area, radial_volume,
                        scale_factors_and_unit_vectors)

FIG_DIR = Path("figures")   # figures worth showing on GitHub
FIG_DIR.mkdir(exist_ok=True)

BLUE, ORANGE = "#2a78d6", "#eb6834"        # categorical slots 1 and 2
GRAY = "#52514e"                           # reference curves and labels

## 5.1 One idea behind all three systems: scale factors

$h_i = \left|\dfrac{\partial\mathbf x}{\partial q_i}\right|$, $\mathbf e_i = \dfrac{1}{h_i}\dfrac{\partial\mathbf x}{\partial q_i}$, and for orthogonal systems
$ds^2 = \sum h_i^2\,dq_i^2$, $dV = h_1h_2h_3\,dq_1dq_2dq_3$, $\nabla = \sum_i \dfrac{\mathbf e_i}{h_i}\dfrac{\partial}{\partial q_i}$.

| System | Coordinates | $(h_1, h_2, h_3)$ | $dV$ |
|---|---|---|---|
| Cartesian | $(x, y, z)$ | $(1, 1, 1)$ | $dx\,dy\,dz$ |
| Cylindrical | $(R, \varphi, z)$ | $(1, R, 1)$ | $R\,dR\,d\varphi\,dz$ |
| Spherical | $(r, \theta, \varphi)$ | $(1, r, r\sin\theta)$ | $r^2\sin\theta\,dr\,d\theta\,d\varphi$ |

*My notes:*

In [ ]:
R, r, theta = sp.symbols("R r theta", positive=True)
x, y, z, phi = sp.symbols("x y z phi", real=True)
systems = {
    "Cartesian": (sp.Matrix([x, y, z]), (x, y, z)),
    "cylindrical": (sp.Matrix([R * sp.cos(phi), R * sp.sin(phi), z]), (R, phi, z)),
    "spherical": (sp.Matrix([r * sp.sin(theta) * sp.cos(phi), r * sp.sin(theta) * sp.sin(phi),
                             r * sp.cos(theta)]), (r, theta, phi)),
}
basis = {}
for name, (position, coords) in systems.items():
    h, e = scale_factors_and_unit_vectors(position, coords)
    basis[name] = (e, coords)
    print(f"{name:11s}: h = {h}, dV = {sp.Mul(*h)} d{coords[0]} d{coords[1]} d{coords[2]}")
    for q, ei in zip(coords, e):
        print(f"             e_{q} = {list(ei)}")

## 5.2 Unit vectors that turn

Cylindrical: $\partial\mathbf e_R/\partial\varphi = \mathbf e_\varphi$, $\partial\mathbf e_\varphi/\partial\varphi = -\mathbf e_R$.
Spherical: $\partial\mathbf e_r/\partial\theta = \mathbf e_\theta$, $\partial\mathbf e_r/\partial\varphi = \sin\theta\,\mathbf e_\varphi$, $\partial\mathbf e_\theta/\partial\theta = -\mathbf e_r$, $\partial\mathbf e_\theta/\partial\varphi = \cos\theta\,\mathbf e_\varphi$, $\partial\mathbf e_\varphi/\partial\varphi = -\sin\theta\,\mathbf e_r - \cos\theta\,\mathbf e_\theta$.

Conventions: $\theta$ is measured from the $+z$ axis (co-latitude) and $\varphi$ is the azimuth; latitude $= 90^\circ - \theta$. On pp. 893-894 of Kundu, read $f(R, \theta, z)$ as $f(R, \varphi, z)$.

*My notes:*

In [ ]:
# Nonzero derivatives of the unit vectors, written in the local basis
for name in ("cylindrical", "spherical"):
    e, coords = basis[name]
    for i, qi in enumerate(coords):
        for j, qj in enumerate(coords):
            comps = derivative_in_basis(e, coords, i, j)
            if any(c != 0 for c in comps):
                terms = " + ".join(f"({c}) e_{coords[k]}" for k, c in enumerate(comps) if c != 0)
                print(f"{name:11s}: d e_{qi} / d {qj} = {terms}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))

ax = axes[0]
phis = np.radians(np.arange(0, 360, 45))
ring = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(ring), np.sin(ring), color=GRAY, lw=0.8)
arrow = dict(angles="xy", scale_units="xy", scale=2.5, width=0.008)
ax.quiver(np.cos(phis), np.sin(phis), np.cos(phis), np.sin(phis), color=BLUE, **arrow)
ax.quiver(np.cos(phis), np.sin(phis), -np.sin(phis), np.cos(phis), color=ORANGE, **arrow)
ax.quiver([0, 0], [0, 0], [1, 0], [0, 1], color=GRAY, **arrow)
ax.text(1.43, 0.06, "e_R", fontsize=9)                 # labels at phi = 0
ax.text(1.06, 0.40, "e_phi", fontsize=9)
ax.text(0.30, -0.14, "e_x", fontsize=9)
ax.text(-0.22, 0.30, "e_y", fontsize=9)
ax.set(xlim=(-1.6, 1.6), ylim=(-1.6, 1.6), aspect="equal", xlabel="x", ylabel="y",
       title="(a) e_R (blue) and e_phi (orange) turn with phi;\ne_x and e_y (gray) do not")

ax = axes[1]
ph = np.linspace(0, 2 * np.pi, 361)
ax.plot(np.degrees(ph), np.cos(ph), color=BLUE, label="R-component: cos(phi)")
ax.plot(np.degrees(ph), -np.sin(ph), "--", color=ORANGE, label="phi-component: -sin(phi)")
ax.set(xlabel="phi (degrees)", ylabel="component of e_x", xticks=range(0, 361, 90),
       title="(b) a constant vector, e_x, has changing\n(R, phi) components")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "ch05_unit_vectors.png", dpi=200)
plt.show()

### Acceleration in cylindrical coordinates

$\dfrac{D\mathbf u}{Dt} = \left(\dfrac{Du_R}{Dt} - \dfrac{u_\varphi^2}{R}\right)\mathbf e_R + \left(\dfrac{Du_\varphi}{Dt} + \dfrac{u_Ru_\varphi}{R}\right)\mathbf e_\varphi + \dfrac{Du_z}{Dt}\,\mathbf e_z$:
centripetal and angular-momentum terms that come from geometry, not from new forces.

*My notes:*

In [ ]:
# Follow one particle along R(t), phi(t), z(t) and differentiate its position twice
t = sp.symbols("t", real=True)
Rt, pt, zt = [sp.Function(n)(t) for n in ("R", "phi", "z")]
e_R = sp.Matrix([sp.cos(pt), sp.sin(pt), 0])
e_phi = sp.Matrix([-sp.sin(pt), sp.cos(pt), 0])
e_z = sp.Matrix([0, 0, 1])
acc = (Rt * e_R + zt * e_z).diff(t, 2)            # acceleration (Cartesian components)
u_R, u_phi = Rt.diff(t), Rt * pt.diff(t)          # velocity components

print("a . e_R   =", sp.simplify(acc.dot(e_R)))     # R'' - R phi'^2
print("a . e_phi =", sp.simplify(acc.dot(e_phi)))   # R phi'' + 2 R' phi'
print("check e_R:  ", sp.simplify(acc.dot(e_R) - (u_R.diff(t) - u_phi**2 / Rt)))        # 0
print("check e_phi:", sp.simplify(acc.dot(e_phi) - (u_phi.diff(t) + u_R * u_phi / Rt)))  # 0
print("check e_z:  ", sp.simplify(acc.dot(e_z) - zt.diff(t, 2)))                        # 0

## 5.3 Surface and volume integrals

On a coordinate surface, the area element is the product of the two scale factors of the coordinates that vary on it,
e.g. $\xi\,d\varphi\,dz$ on the cylinder $R = \xi$ and $\xi^2\sin\theta\,d\theta\,d\varphi$ on the sphere $r = \xi$.

Volumes of simple landforms: a cone has $V = \tfrac13\pi R_b^2H$; a Gaussian hill $z = He^{-R^2/\ell^2}$ has $V = \pi\ell^2H$.

*My notes:*

In [ ]:
Rg = np.linspace(0, 5000, 50001)                    # cone: H = 1 km, basal radius 5 km
cone = radial_volume(1000 * (1 - Rg / 5000), Rg)
print(f"cone:          {cone / 1e9:.2f} km^3 (exact {np.pi * 5**2 * 1 / 3:.2f})")

Rg = np.linspace(0, 1000, 100001)                    # Gaussian hill: H = 50 m, l = 100 m
T = 50 * np.exp(-Rg**2 / 100**2)
print(f"Gaussian hill: {radial_volume(T, Rg):.4g} m^3 (exact {np.pi * 100**2 * 50:.4g})")
wrong = 2 * np.pi * trapezoid(T, Rg)                # the same integral without the factor R
print(f"forgetting R:  {wrong:.4g} -- wrong, and its units are m^2, not m^3")

## 5.4 Research connection: a round planet and radial landforms

- **Area on the sphere:** $dA = R_E^2\cos\vartheta\,d\vartheta\,d\lambda$, so a cell between latitudes $\vartheta_1$ and $\vartheta_2$ has area $R_E^2\,\Delta\lambda\,(\sin\vartheta_2 - \sin\vartheta_1)$. Global averages on latitude-longitude grids must be area-weighted.
- **Radial landforms:** fans, deltas, volcanoes, craters, wells, rotating tanks; a flux spreading radially is diluted as $1/R$.
- **Rotation:** $f = 2\Omega\sin\vartheta = 2\boldsymbol\Omega\cdot\mathbf e_r$.

*My notes:*

In [ ]:
lat = np.arange(-89.5, 90.0, 1.0)                    # centres of 1-degree latitude bands
area = latlon_cell_area(lat - 0.5, lat + 0.5, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
axes[0].plot(lat, area, color=BLUE)
for la, name in [(0.0, "equator"), (36.0, "36 N (Durham)"), (80.0, "80 N")]:
    a_cell = latlon_cell_area(la - 0.5, la + 0.5, 1.0)
    axes[0].plot(la, a_cell, "o", color=BLUE, ms=6)
    side = {0.0: ("center", (0, 8)), 36.0: ("right", (-8, -14)), 80.0: ("right", (-10, 8))}[la]
    axes[0].annotate(f"{name}: {a_cell:,.0f} km^2", (la, a_cell), textcoords="offset points",
                     xytext=side[1], ha=side[0], fontsize=8)
axes[0].set(xlim=(-90, 90), ylim=(0, 14000), xlabel="latitude (degrees)",
            ylabel="area of a 1 x 1 degree cell (km^2)", title="(a) grid cells shrink toward the poles")

field = 1 + 2 * np.sin(np.radians(lat)) ** 2          # a quantity that grows poleward
naive, weighted = field.mean(), area_weighted_mean(field, lat, 1.0)
axes[1].plot(lat, field, color=GRAY, lw=1, label="field = 1 + 2 sin^2(latitude)")
axes[1].axhline(naive, color=ORANGE, ls="--", label=f"equal weights: {naive:.2f}")
axes[1].axhline(weighted, color=BLUE, label=f"area weighted: {weighted:.2f}")
axes[1].set(xlim=(-90, 90), xlabel="latitude (degrees)", ylabel="value",
            title="(b) equal weights over-count the poles")
axes[1].legend(fontsize=8, loc="upper center")
fig.tight_layout()
fig.savefig(FIG_DIR / "ch05_round_earth.png", dpi=200)
plt.show()
print(f"total area check: {360 * area.sum():.4e} km^2 vs 4 pi R^2 = {4 * np.pi * 6371.0**2:.4e}")

In [ ]:
durham = (35.994, -78.899)
hatteras = (35.2505, -75.5288)                        # Cape Hatteras lighthouse
print(f"Durham -> Cape Hatteras: {great_circle_distance(*durham, *hatteras):.0f} km")
for la in (0, 36, 75):
    print(f"1 degree of longitude at {la:2d} N: {great_circle_distance(la, 0, la, 1):6.1f} km")

f36 = coriolis_parameter(durham[0])
print(f"Coriolis parameter at Durham: {f36:.2e} 1/s; inertial period {2 * np.pi / f36 / 3600:.1f} h")
for U, L, what in [(1.0, 1e3, "small river"), (1.0, 1e4, "large river"), (0.1, 1e5, "lake or shelf")]:
    print(f"Rossby number U/(f L), {what:13s} (U = {U} m/s, L = {L / 1e3:g} km): {U / (f36 * L):.2f}")

## Exercises 5.1-5.6

*Work space: solve each exercise here, then check against Appendix C of the study guide.*